# Language Processing Laboratory — Session 01
## Foundations for Persian NLP: Jupyter, UTF-8, Unicode, and Data Hygiene

**Duration:** 120 minutes  
**Level:** Undergraduate  
**Prerequisite:** Basic Python programming  
**Libraries required today:** Python standard library only

> **هدف جلسه:** پیش از ساخت هر مدل زبانی، باید بدانیم «متن» در رایانه دقیقاً چگونه نمایش داده می‌شود. در فارسی، تفاوت نویسه‌های عربی و فارسی، نیم‌فاصله، ارقام، فاصله‌ها و نرمال‌سازی Unicode می‌توانند مستقیماً روی واژگان، شمارش‌ها و کیفیت مدل اثر بگذارند.

This is a **student starter notebook**. Complete every section marked **✍️ Exercise** and every `TODO` before submission.

### Repository and submission note

- This is the **starter notebook** for the currently released lab.
- Read `docs/SETUP_FA.md` before starting if your Jupyter/Python environment is not ready.
- **Do not publish your completed solution notebook in a public GitHub fork/repository.** Submit it only through the channel announced by the instructor.
- If you use a non-trivial external source or AI assistant, follow `docs/AI_USE_POLICY_FA.md` and disclose it in the final notebook section.
- Keep the original exercise headings and function names so your submission remains easy to review.


## Learning outcomes

By the end of this lab, you should be able to:

1. Run and document a reproducible Python/Jupyter experiment.
2. Read, write, encode, and decode Persian text using UTF-8.
3. Explain the difference between **Unicode normalization** and **Persian linguistic normalization**.
4. Detect common Arabic/Persian character variants such as `ي/ی` and `ك/ک`.
5. Explain why the **Zero Width Non-Joiner (ZWNJ, U+200C)** matters in Persian.
6. Implement a small, explicit, testable Persian text-normalization function.
7. Audit a small Persian corpus and quantify how normalization changes its vocabulary.

### What is *not* the goal today
We are **not** trying to build a perfect Persian tokenizer or a full NLP pipeline. Today is about the representation and quality of text *before* modeling.

## Lab workflow and grading

Suggested pace:

| Time | Activity |
|---:|---|
| 0–10 min | Environment check + notebook conventions |
| 10–25 min | Python strings, UTF-8, encode/decode |
| 25–45 min | Unicode and Persian/Arabic character variants |
| 45–60 min | ZWNJ and spacing |
| 60–85 min | Build a small Persian normalizer |
| 85–105 min | Corpus audit + statistics |
| 105–115 min | Interpretation and reflection |
| 115–120 min | Final checks + submission |

**Rubric (10 points)**

- 1 pt — reproducible notebook and student metadata
- 2 pts — Unicode/UTF-8 exercises and explanations
- 3 pts — normalization function and public checks
- 2 pts — corpus audit/statistics
- 2 pts — interpretation, reflection, and clean submission

### Submission convention
Rename your completed file to:

`STUDENT_ID_session01.ipynb`

Your notebook must run from top to bottom without syntax errors.

## 0. Reproducibility check

A language-processing experiment is only useful if someone else can reproduce it. Run the following cell **without changing it**.

In [ ]:
import sys
import platform
import re
import unicodedata
from collections import Counter

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Unicode database version:', unicodedata.unidata_version)

if sys.version_info < (3, 10):
    print('WARNING: Python 3.10+ is recommended for this course.')
else:
    print('Environment check: OK')

### ✍️ Exercise 0 — Student metadata

Complete the cell below. Use Persian text where appropriate.

In [ ]:
# TODO
student_id = ''
student_name = ''
university = ''

print('Student ID:', student_id)
print('Name:', student_name)
print('University:', university)

---
# 1. Python strings and UTF-8

Python 3 strings are Unicode strings. But files and network messages are stored as **bytes**, so encoding still matters.

Run the example below and inspect the difference between a Python `str` and UTF-8 `bytes`.

In [ ]:
text = 'پردازش زبان طبیعی'
utf8_bytes = text.encode('utf-8')
round_trip = utf8_bytes.decode('utf-8')

print('String:', text)
print('Type:', type(text))
print('Python length:', len(text))
print('UTF-8 bytes:', utf8_bytes)
print('Number of UTF-8 bytes:', len(utf8_bytes))
print('Round trip preserved text:', round_trip == text)

### Key idea

`len(text)` counts Python Unicode code points, while `len(text.encode('utf-8'))` counts bytes. For Persian text, these numbers are usually different.

### ✍️ Exercise 1 — UTF-8 round trip

Create a Persian sentence about NLP. Then:

1. print the sentence;
2. print its Python length;
3. encode it as UTF-8;
4. print the byte length;
5. decode it back;
6. verify that the original and decoded strings are equal.

In [ ]:
# TODO: Exercise 1
sentence = ''

# Write your code below

**Short answer (1–2 sentences):** Why is explicitly using UTF-8 important when exchanging Persian text between systems?

> TODO: write your answer here.

---
# 2. Unicode: visually similar does not mean computationally identical

Persian text often contains characters inherited from Arabic keyboards, copied web pages, PDFs, or legacy systems.

Two characters can **look similar** but have different Unicode code points.

In [ ]:
def show_codepoints(text):
    """Display index, character, Unicode code point, and Unicode name."""
    for i, ch in enumerate(text):
        print(f'{i:>2} | {repr(ch):>6} | U+{ord(ch):04X} | {unicodedata.name(ch, "UNKNOWN")}')

for sample in ['ی', 'ي', 'ک', 'ك']:
    show_codepoints(sample)
    print('-' * 70)

### Canonical Unicode normalization

Unicode can sometimes represent the **same abstract character** in more than one way. For example, `آ` can appear as one precomposed code point or as `ا` followed by a combining mark.

`unicodedata.normalize('NFC', text)` performs canonical composition where possible.

In [ ]:
precomposed = 'آ'
decomposed = 'ا\u0653'  # ALEF + MADDA ABOVE

print('Looks similar:')
print(precomposed, decomposed)
print('Equal before NFC?', precomposed == decomposed)
print('Lengths:', len(precomposed), len(decomposed))

pre_nfc = unicodedata.normalize('NFC', precomposed)
dec_nfc = unicodedata.normalize('NFC', decomposed)
print('Equal after NFC?', pre_nfc == dec_nfc)

print('\nCode points before normalization:')
show_codepoints(decomposed)
print('\nCode points after NFC:')
show_codepoints(dec_nfc)

### Important distinction

**NFC does not convert Arabic `ي` to Persian `ی`, or Arabic `ك` to Persian `ک`.** Those are different letters/code points, not merely two canonical encodings of the same character. That conversion is a **language-specific normalization policy**.

This distinction matters in research: every normalization rule should have a reason and should be documented.

### ✍️ Exercise 2 — Character audit

Inspect the mixed-quality sentence below and count each of the following:

- Persian `ی`
- Arabic `ي`
- Persian `ک`
- Arabic `ك`

Do not normalize the sentence yet.

In [ ]:
mixed_text = 'يک دانشجوی فارسی می‌تواند روی متن فارسي و كدهای متفاوت کار کند.'

# TODO: Exercise 2
variant_counts = {
    'ی': 0,
    'ي': 0,
    'ک': 0,
    'ك': 0,
}

# Replace the zeros above by computed counts.
print(variant_counts)

---
# 3. ZWNJ (نیم‌فاصله) is a real character

Persian uses the **Zero Width Non-Joiner**, Unicode `U+200C`, in many orthographic contexts.

Compare:

- `میروم`
- `می روم`
- `می‌روم`

The third form contains a ZWNJ between `می` and `روم`.

In [ ]:
forms = ['میروم', 'می روم', 'می‌روم']

for form in forms:
    print(f'{repr(form):12} | length={len(form)} | split()={form.split()}')

print('\nCode points for the ZWNJ form:')
show_codepoints('می‌روم')

### ✍️ Exercise 3 — Reason about spacing

Answer in 2–4 sentences:

1. Why does `str.split()` treat `می روم` differently from `می‌روم`?
2. Why could inconsistent ZWNJ usage increase the apparent vocabulary size of a corpus?
3. Why might replacing every ZWNJ with a regular space be an undesirable “fix”?

> TODO: write your answer here.

---
# 4. Build an explicit Persian normalization policy

A good normalizer should be:

- **explicit** — we know which transformations it performs;
- **deterministic** — the same input gives the same output;
- **testable** — expected behavior can be checked;
- **task-aware** — we avoid deleting potentially useful information without a reason.

For this first lab, implement only a **minimal, conservative** normalizer.

### Required policy

Your function must:

1. apply Unicode NFC;
2. convert Arabic `ي` and `ى` to Persian `ی`;
3. convert Arabic `ك` to Persian `ک`;
4. collapse repeated ordinary whitespace to one space;
5. strip leading/trailing whitespace;
6. preserve ZWNJ (`U+200C`);
7. preserve punctuation and digits.

> We deliberately preserve punctuation and digits because they may be useful features in downstream tasks. Normalization is not the same thing as “delete everything unusual.”

In [ ]:
def normalize_persian_basic(text):
    """Return a minimally normalized Persian string.

    Required behavior:
      - NFC Unicode normalization
      - Arabic ي/ى -> Persian ی
      - Arabic ك -> Persian ک
      - collapse ordinary whitespace
      - strip leading/trailing whitespace
      - preserve ZWNJ, punctuation, and digits
    """
    # TODO: Exercise 4
    return text

### Public checks

These tests do not cover every edge case, but your function should pass them before submission.

In [ ]:
def run_public_checks(normalize_fn):
    cases = [
        ('يك متن فارسي', 'یک متن فارسی'),
        ('  پردازش   زبان  ', 'پردازش زبان'),
        ('می‌روم', 'می‌روم'),  # ZWNJ must remain
        ('نسخه 2، آماده است!', 'نسخه 2، آماده است!'),  # digits/punctuation remain
        ('ا\u0653زمایش', 'آزمایش'),  # NFC composition
    ]

    passed = 0
    for source, expected in cases:
        actual = normalize_fn(source)
        ok = actual == expected
        print(('PASS' if ok else 'FAIL'), '|', repr(source), '->', repr(actual))
        if not ok:
            print('      expected:', repr(expected))
        passed += int(ok)

    print(f'\n{passed}/{len(cases)} public checks passed.')
    return passed == len(cases)

# Run this after implementing normalize_persian_basic:
# run_public_checks(normalize_persian_basic)

### ✍️ Exercise 4b — Explain your policy

In 2–3 sentences, explain why the lab asks you to **preserve punctuation and digits** instead of deleting them automatically.

> TODO: write your answer here.

---
# 5. Corpus audit: quantify data-quality problems

Real corpora are rarely clean. A useful NLP workflow measures the problem **before** changing the data.

The mini-corpus below intentionally contains several inconsistencies. Keep it unchanged.

In [ ]:
raw_corpus = [
    'پردازش زبان طبیعی یکی از شاخه‌های هوش مصنوعی است.',
    'يادگيري ماشين در پردازش متن كاربرد دارد.',
    'دانشجوها می توانند با داده‌های متنی کار کنند.',
    'قيمت اين كتاب 250000 تومان است.',
    'نسخه ۲ نرم‌افزار امروز منتشر شد!!!',
    'من نمی‌دانم چرا نیم‌فاصله مهم است؟',
    'آزمايشگاه زبان، محيطی برای تجربه عملی است.',
    'مدل‌های زبانی با متن فارسی و فارسي روبه‌رو می‌شوند.',
    '  داده   خوب،   پیش‌نیاز   مدل   خوب   است.  ',
]

for i, line in enumerate(raw_corpus, start=1):
    print(f'{i:02d}: {line}')

### ✍️ Exercise 5 — Write a data-quality audit

Implement `audit_persian_text(text)` so that it returns a dictionary with at least these counts:

- `arabic_yeh` — number of `ي`
- `farsi_yeh` — number of `ى`
- `arabic_kaf` — number of `ك`
- `zwnj` — number of `U+200C`
- `latin_digits` — number of `0–9`
- `persian_digits` — number of `۰–۹`
- `replacement_char` — number of Unicode replacement characters `�` (`U+FFFD`)

Use the standard library only.

In [ ]:
def audit_persian_text(text):
    # TODO: Exercise 5
    return {
        'arabic_yeh': 0,
        'farsi_yeh': 0,
        'arabic_kaf': 0,
        'zwnj': 0,
        'latin_digits': 0,
        'persian_digits': 0,
        'replacement_char': 0,
    }

corpus_text = '\n'.join(raw_corpus)
print(audit_persian_text(corpus_text))

### ✍️ Exercise 6 — Before/after vocabulary statistics

Use **whitespace splitting only for this experiment**. It is intentionally simplistic.

For both the raw corpus and normalized corpus, compute:

- total token count (**tokens**);
- unique token count (**types**);
- type–token ratio (`types / tokens`);
- 10 most frequent tokens.

Then answer: **Did normalization reduce the number of unique tokens? Why or why not?**

In [ ]:
def whitespace_tokens(lines):
    return ' '.join(lines).split()

# TODO: Exercise 6
raw_tokens = []
normalized_corpus = []
normalized_tokens = []

# Compute the requested statistics below.

**Interpretation (3–5 sentences):**

> TODO: compare the raw and normalized corpus. Do not merely report numbers; explain what changed and why.

---
# 6. A critical NLP lesson: normalization can help *and* harm

There is no universal rule that says “more normalization is always better.”

Examples:

- punctuation may help sentiment or dialogue modeling;
- digits may carry prices, dates, versions, or measurements;
- ZWNJ may encode meaningful Persian orthographic structure;
- repeated characters can express emphasis in social-media text;
- emojis may be predictive for sentiment.

A professional NLP pipeline therefore records **what was changed, why, and for which task**.

### ✍️ Exercise 7 — Normalization decision

Suppose you are building each of the following systems. For each one, write **one feature you would preserve** and **one transformation you might consider**:

1. Persian sentiment analysis on social-media posts
2. Persian news topic classification
3. Persian search engine for product prices

> TODO: write your answers here.

---
# 7. Exit ticket

Answer briefly, in your own words.

1. What is the difference between **UTF-8** and **Unicode**?
2. Why is NFC normalization not sufficient for normalizing Persian `ي/ی` and `ك/ک`?
3. What is ZWNJ, and why can it affect NLP statistics?
4. Give one example of a normalization step that could remove useful information for some tasks.

> TODO: write your answers here.

## Optional challenge — digit normalization

This challenge is **not required** for full credit.

Implement a function that converts both Western digits (`0123456789`) and Arabic-Indic digits (`٠١٢٣٤٥٦٧٨٩`) to Persian digits (`۰۱۲۳۴۵۶۷۸۹`) using `str.translate`.

Then explain one task where digit normalization is helpful and one task where preserving the original digit style might matter.

In [ ]:
def to_persian_digits(text):
    # OPTIONAL TODO
    return text

# Example target behavior:
# to_persian_digits('نسخه 2 در سال ٢٠٢٦') -> 'نسخه ۲ در سال ۲۰۲۶'

---
# Final submission checklist

Before submitting, confirm that:

- [ ] Exercise 0 contains your information.
- [ ] Exercises 1–7 are complete.
- [ ] `normalize_persian_basic` passes all public checks.
- [ ] Your audit function computes values rather than returning hard-coded counts.
- [ ] Your explanations are written in your own words.
- [ ] You ran **Kernel → Restart Kernel and Run All Cells** (or equivalent).
- [ ] The notebook finishes without syntax errors.
- [ ] The file is named `STUDENT_ID_session01.ipynb`.

### Academic practice
You may discuss concepts with classmates, but your submitted code and written explanations must reflect your own understanding. If you use an external source or AI assistant for a non-trivial idea, cite it briefly in a final note below.

**External help / sources used (if any):**

> TODO / None

## Reference points for this lab

- Dan Jurafsky & James H. Martin, *Speech and Language Processing*, text-processing/tokenization chapters.
- Stanford CS124, early-course setup and regular-expression/tokenization programming materials.
- Python `unicodedata` documentation.
- Hazm Persian NLP Toolkit documentation (for comparison with production-grade Persian normalization; not required in this session).

**End of Session 01.**